<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Tokenización por Subpalabras: BPE y WordPiece</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Cuando la tokenización por palabras resulta insuficiente, aparecen estrategias por subpalabras como BPE y WordPiece. Este notebook introduce esa solución para manejar vocabularios grandes y palabras no vistas.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Gran parte de los modelos modernos de NLP dependen de tokenizadores que equilibran tamaño de vocabulario y capacidad de generalización.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender por qué existen las subpalabras y qué problema intentan resolver BPE y WordPiece.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Dividir mejor el texto no significa fragmentarlo al azar, sino buscar unidades reutilizables y eficientes.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Una subpalabra es un fragmento intermedio entre carácter y palabra completa que puede recombinarse en muchos contextos.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición del texto
"""Este bloque define un texto de entrada que será segmentado mediante tokenización por subpalabras.
La variable "texto" contiene una cadena que será dividida en tokens, algunos de ellos menores que las palabras completas.
El contenido permitirá observar cómo BPE y WordPiece fragmentan el texto en subunidades reutilizables."""

texto = "La tokenización por subpalabras permite manejar vocabularios amplios y palabras desconocidas."

texto # Muestra el texto original para verificar su contenido.


In [ ]:
# Bloque 2 — Tokenización WordPiece con HuggingFace
"""Este bloque aplica tokenización WordPiece utilizando un tokenizador preentrenado.
Este bloque importa "AutoTokenizer" de "transformers" para utilizar un tokenizador preentrenado de HuggingFace.

Aplicamos la clase "AutoTokenizer" acompañada del método ".from_pretrained()" a la variable "tokenizer_wp" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizer_wp" almacena el objeto tokenizador preentrenado cargado.

Aplicamos el método ".tokenize()" al objeto tokenizador preentrenado para obtener la lista de tokens subword correspondientes al texto de entrada.
El argumento "texto" es el texto de entrada que se desea tokenizar.

La variable "tokens_wordpiece" almacena los tokens generados por el tokenizador.
WordPiece fragmenta palabras en unidades frecuentes para reducir la aparición de tokens desconocidos."""

from transformers import AutoTokenizer

tokenizer_wp = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens_wordpiece = tokenizer_wp.tokenize(texto)

tokens_wordpiece # Muestra la lista de tokens generados por el tokenizador WordPiece.



In [ ]:
# Bloque 3 — Conversión de tokens WordPiece a IDs
"""Este bloque pasa la lista de tokens WordPiece al método "encode()" del tokenizador.
El argumento "tokens_wordpiece" es la lista de tokens obtenida en el bloque anterior.
Con la versión instalada de Transformers, el método interpreta la lista como un lote y vuelve a tokenizar cada elemento por separado.
El parámetro "add_special_tokens=True" agrega tokens especiales a cada secuencia del lote.

La variable "ids_wordpiece" almacena una lista de secuencias de IDs, no una única secuencia del texto original.
Para obtener los IDs de los tokens ya generados puede utilizarse "convert_tokens_to_ids()"; para codificar el texto completo, "encode(texto)"."""

ids_wordpiece = tokenizer_wp.encode(tokens_wordpiece, add_special_tokens=True)

ids_wordpiece # Muestra las secuencias de IDs generadas a partir de cada elemento de la lista WordPiece.


In [ ]:
# Bloque 4 — Tokenización BPE con GPT-2
"""Este bloque aplica tokenización BPE utilizando el tokenizador de GPT-2.
Aplicamos la clase "AutoTokenizer" acompañada del método ".from_pretrained()" a la variable "tokenizer_bpe" para cargar un tokenizador preentrenado.
El argumento "gpt2" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizer_bpe" almacena el objeto tokenizador preentrenado cargado.

Aplicamos el método ".tokenize()" al objeto tokenizador preentrenado para obtener la lista de tokens subword correspondientes al texto de entrada.
El argumento "texto" es el texto de entrada que se desea tokenizar.

La variable "tokens_bpe" almacena los tokens generados por el tokenizador.
BPE segmenta palabras en subunidades aprendidas a partir de estadísticas de frecuencia."""

tokenizer_bpe = AutoTokenizer.from_pretrained("gpt2")
tokens_bpe = tokenizer_bpe.tokenize(texto)

tokens_bpe  # Muestra la lista de tokens generados por el tokenizador BPE.


In [ ]:
# Bloque 5 — Conversión de tokens BPE a IDs
"""Este bloque pasa la lista de tokens BPE al método "encode()" del tokenizador.
El argumento "tokens_bpe" es la lista de tokens obtenida en el bloque anterior.
Con la versión instalada de Transformers, el método interpreta la lista como un lote y vuelve a tokenizar cada elemento por separado.
El parámetro "add_special_tokens=False" indica que no se agregan tokens especiales.

La variable "ids_bpe" almacena una lista de secuencias de IDs, no una única secuencia del texto original.
Para obtener los IDs de los tokens ya generados puede utilizarse "convert_tokens_to_ids()"; para codificar el texto completo, "encode(texto)"."""

ids_bpe = tokenizer_bpe.encode(tokens_bpe, add_special_tokens=False)

ids_bpe  # Muestra las secuencias de IDs generadas a partir de cada elemento de la lista BPE.


In [ ]:
# Bloque 6 — Comparación de longitudes
"""Este bloque compara las longitudes de las listas externas generadas por las llamadas a "encode()".
La variable "longitudes" almacena en forma de diccionario el número de secuencias de cada lote.
Estas longitudes reflejan el número de elementos pasados a cada llamada, no el número total de IDs ni una comparación equivalente de secuencias completas."""

longitudes = {
    "longitud_wordpiece": len(ids_wordpiece),
    "longitud_bpe": len(ids_bpe)
}

longitudes # Muestra el diccionario con las longitudes de los lotes WordPiece y BPE.

In [ ]:
# Bloque 7 — Padding para longitud fija
"""Este bloque aplica truncado y padding a las listas externas de secuencias WordPiece y BPE.
La variable "longitud_objetivo" define la longitud deseada de cada lista externa, que en este caso es 20.

En el primer caso aplicamos slicing sobre la variable "ids_wordpiece" para limitar su longitud a "longitud_objetivo".
El rango [:longitud_objetivo] indica que se tomarán los elementos desde el índice cero hasta el índice "longitud_objetivo" (sin incluir este último).
La variable "secuencia_wp_fija" almacena la lista externa ajustada a la longitud deseada.

Utilizamos el operador de asignación "+=" para añadir ceros hasta alcanzar la longitud objetivo.
La expresión "[0] * (longitud_objetivo - len(secuencia_wp_fija))" genera una lista de ceros cuya longitud es la diferencia entre la longitud objetivo
y la longitud actual de la secuencia.

En el segundo caso aplicamos slicing sobre la variable "ids_bpe" para limitar su longitud a "longitud_objetivo".
El rango [:longitud_objetivo] indica que se tomarán los elementos desde el índice cero hasta el índice "longitud_objetivo" (sin incluir este último).
La variable "secuencia_bpe_fija" almacena la lista externa ajustada a la longitud deseada.

Utilizamos el operador de asignación "+=" para añadir ceros hasta alcanzar la longitud objetivo.
La expresión "[0] * (longitud_objetivo - len(secuencia_bpe_fija))" genera una lista de ceros cuya longitud es la diferencia entre la longitud objetivo
y la longitud actual de la secuencia.

La operación no ajusta las secuencias internas ni crea máscaras de atención; en este ejemplo ambas listas externas se recortan a 20 sin añadir ceros.
Si se añadieran ceros a BPE, no serían un token de padding configurado: el tokenizador GPT-2 no lo tiene definido por defecto."""

longitud_objetivo = 20

secuencia_wp_fija = ids_wordpiece[:longitud_objetivo]
secuencia_wp_fija += [0] * (longitud_objetivo - len(secuencia_wp_fija))

secuencia_bpe_fija = ids_bpe[:longitud_objetivo]
secuencia_bpe_fija += [0] * (longitud_objetivo - len(secuencia_bpe_fija))


secuencia_wp_fija, secuencia_bpe_fija # Muestra las listas externas tras el truncado y el posible padding.


In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del proceso de tokenización por subpalabras.
La variable "resultado" almacena en forma de diccionario los tokens, los lotes de IDs, las longitudes y las listas externas ajustadas.
La estructura permite visualizar las diferencias entre ambos métodos y sus representaciones numéricas."""

resultado = {
    "tokens_wordpiece": tokens_wordpiece,
    "ids_wordpiece": ids_wordpiece,
    "tokens_bpe": tokens_bpe,
    "ids_bpe": ids_bpe,
    "longitudes": longitudes,
    "secuencia_wp_fija": secuencia_wp_fija,
    "secuencia_bpe_fija": secuencia_bpe_fija
}

resultado # Muestra el diccionario con los resultados finales.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto por qué los modelos modernos suelen preferir unidades de subpalabras frente a palabras completas fijas.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">BPE y WordPiece ayudan a tratar mejor palabras raras, variaciones y vocabularios extensos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Pensar que más fragmentación siempre es mejor sin considerar legibilidad y eficiencia del vocabulario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en los Embeddings para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en los Embeddings y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">02_Embeddings/01_Los_embeddings.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">